In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [2]:
import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

# from model import SeqNN
from akita_model.model import SeqNN

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(device)

cuda:0


In [4]:
model = SeqNN()
model.load_state_dict(torch.load("/home1/smaruj/pytorch_akita/models/finetuned/mouse/Hsieh2019_mESC/checkpoints/Akita_v2_mouse_Hsieh2019_mESC_model0_finetuned.pth", map_location=device))
model.eval()

SeqNN(
  (stochastic_reverse_complement): StochasticReverseComplement()
  (stochastic_shift): StochasticShift()
  (conv_block_1): ConvBlock(
    (conv): Conv1d(4, 128, kernel_size=(15,), stride=(1,), padding=(7,), bias=False)
    (batch_norm): BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_tower): ConvTower(
    (conv_tower): Sequential(
      (0): ReLU()
      (1): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ReLU()
      (5): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (7): MaxPool1d(kernel_size=2, stride=2, paddi

In [5]:
import pandas as pd

In [6]:
FOLD = 0
input_dir = "/scratch1/smaruj/genomic_map_transformation"

In [ ]:
# df = pd.read_csv(
#     "/project2/fudenber_735/tensorflow_models/akita/v2/data/mm10/sequences.bed",
#     sep="\t",
#     header=None,
#     names=["chrom", "start", "end", "fold"]
# )

In [ ]:
# df_fold0 = df[df["fold"] == "fold0"]

# df_fold0.to_csv(
#     "/scratch1/smaruj/genomic_map_transformation/df_select_fold0.tsv",
#     sep="\t",
#     index=False,
#     header=True
# )

In [7]:
df = pd.read_csv(f"{input_dir}/df_select_fold{FOLD}.tsv", sep="\t")

In [ ]:
df = df[:10]

In [8]:
df['target_chrom'] = df['chrom'].shift(-1)
df['target_start'] = df['start'].shift(-1)
df['target_end'] = df['end'].shift(-1)

In [9]:
# Fill last row with values from the first row
df.loc[df.index[-1], 'target_chrom'] = df.loc[df.index[0], 'chrom']
df.loc[df.index[-1], 'target_start'] = df.loc[df.index[0], 'start']
df.loc[df.index[-1], 'target_end'] = df.loc[df.index[0], 'end']

In [10]:
# Convert to int
df['target_start'] = df['target_start'].astype(int)
df['target_end'] = df['target_end'].astype(int)

In [12]:
df.to_csv(
    "/scratch1/smaruj/genomic_map_transformation/genomic_optimization_fold0.tsv",
    sep="\t",
    index=False,
    header=True
)

In [ ]:
# to ensure the local, forked ledidi is used
# not the one installed using pip

import sys
sys.path.insert(0, "/home1/smaruj/ledidi/ledidi/")  # Add the directory where "ledidi" is located
from ledidi_whole_seq import Ledidi

In [ ]:
df["last_accepted_step"] = -1  # initialize with placeholder

In [ ]:
df

In [ ]:
for i, row in enumerate(df[1:2].itertuples(index=False)):
    chrom, pred_start, pred_end = row.chrom, row.start, row.end
    target_chrom, target_start, target_end = row.target_chrom, row.target_start, row.target_end
    
    print(f"Map transformation starting from genome location: {chrom}:{pred_start}-{pred_end}")
    print(f"To a genome location: {target_chrom}:{target_start}-{target_end}")
    
    X = torch.load(f"/scratch1/smaruj/genomic_map_transformation/ohe_X_fold0/{chrom}_{pred_start}_{pred_end}_X.pt", weights_only=True, map_location=device)
    target = torch.load(f"/scratch1/smaruj/genomic_map_transformation/genomic_targets_fold0/{target_chrom}_{target_start}_{target_end}_target.pt", weights_only=True, map_location=device)

    wrapper = Ledidi(model, 
                    input_loss=torch.nn.L1Loss(reduction='sum'), 
                    output_loss=torch.nn.L1Loss(reduction='sum'),   # output_loss=torch.nn.MSELoss(),
                    batch_size=1,
                    l=0.05,
                    max_iter=2000,
                    early_stopping_iter=2000,
                    return_history=False,
                    verbose=True
                    ).cuda()
    
    generated_seq, last_update = wrapper.fit_transform(X, target)
    
    # Update df with last_accepted_step
    df.at[i, "last_accepted_step"] = last_update
    
    # torch.save(generated_seq, f"/scratch1/smaruj/genomic_insertion_loci/genomic_optimization_results/fold{FOLD}/{chrom}_{pred_start}_{pred_end}_seq.pt")

# df.to_csv(f"/scratch1/smaruj/genomic_insertion_loci/genomic_optimization_results/fold{FOLD}_test.tsv", sep="\t", index=False)

In [ ]:
# slice_torch = X[:,:,start_index : end_index]

In [ ]:
# X_l_flank = X[:,:,start_index - 2048*2 : start_index]
# X_r_flank = X[:,:,end_index : end_index + 2048*2]

In [ ]:
# x_bar, history = wrapper.fit_transform(X, modified_vector_tensor)
# if use_semifreddo=True
# x_bar = wrapper.fit_transform(slice_torch, y_bar, X_l_flank=X_l_flank, X_r_flank=X_r_flank)
# x_bar, history = wrapper.fit_transform(slice_torch, y_bar, X_l_flank=X_l_flank, X_r_flank=X_r_flank)

In [ ]:
# if use_semifreddo=False
# x_bar = wrapper.fit_transform(X, y_bar)
x_bar, history = wrapper.fit_transform(X, y_bar)

In [ ]:
x_bar

## Input and Output Loss Plots

In [ ]:
%matplotlib inline
import numpy
import matplotlib.pyplot as plt
import seaborn; seaborn.set_style('whitegrid')

plt.figure(figsize=(5, 3))
plt.plot(history['input_loss'], c='0.7', label="Input Loss")
plt.legend(fontsize=10)
plt.xlabel("Iterations")
plt.ylabel("Loss")

seaborn.despine(left=True, bottom=True)
plt.show()

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(history['output_loss'], c='0.3', label="Output Loss")
plt.legend(fontsize=10)
plt.xlabel("Iterations")
plt.ylabel("Loss")

seaborn.despine(left=True, bottom=True)
plt.show()

In [ ]:
seq1_indices = torch.argmax(generated_seq, dim=1)
seq2_indices = torch.argmax(X, dim=1)

In [ ]:
num_differences = (seq1_indices != seq2_indices).sum().item()

print(f"Number of differing nucleotides: {num_differences}")

## Predicted Maps

In [ ]:
model.eval()
with torch.no_grad():
    pred = model(generated_seq)
    pred_back = model(X)

In [ ]:
pred.shape, pred_back.shape

In [ ]:
from scipy.stats import pearsonr

# Flatten tensors
x_flat = pred.view(-1).cpu().numpy()
y_flat = target.view(-1).cpu().numpy()

# Calculate Pearson R
r_value, _ = pearsonr(x_flat, y_flat)
print(f"Pearson R: {r_value}")

In [ ]:
import numpy as np

In [ ]:
# Helper function to set diagonal elements to a specific value
def set_diag(matrix, value, k):
    # Explicitly set the diagonal to 'value' (in this case, np.nan) for each k
    rows, cols = matrix.shape
    for i in range(rows):
        if 0 <= i + k < cols:
            matrix[i, i + k] = value


def from_upper_triu(vector_repr, matrix_len, num_diags):
    # Ensure vector_repr is a NumPy array (if it's a PyTorch tensor, convert it)
    if isinstance(vector_repr, torch.Tensor):
        vector_repr = vector_repr.detach().flatten().cpu().numpy()  # Flatten and convert to NumPy array

    # Initialize a zero matrix of shape (matrix_len, matrix_len)
    z = np.zeros((matrix_len, matrix_len))

    # Get the indices for the upper triangular matrix
    triu_tup = np.triu_indices(matrix_len, num_diags)

    # Assign the values from the vector_repr to the upper triangular part of the matrix
    z[triu_tup] = vector_repr

    # Set the diagonals specified by num_diags to np.nan
    for i in range(-num_diags + 1, num_diags):
        set_diag(z, np.nan, i)

    # Ensure the matrix is symmetric
    return z + z.T

In [ ]:
matrix_to_plot = from_upper_triu(pred[0, 0, :], matrix_len=512, num_diags=2)

In [ ]:
matrix_og_to_plot = from_upper_triu(pred_back[0, 0, :], matrix_len=512, num_diags=2)

In [ ]:
matrix_to_plot_genomic = from_upper_triu(target[0, 0, :], matrix_len=512, num_diags=2)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.figure(figsize=(8, 8))
plt.matshow(matrix_og_to_plot.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
plt.colorbar()
plt.savefig("original_map.svg", format='svg')
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
plt.matshow(matrix_to_plot_genomic.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
plt.colorbar()
plt.savefig("target_example_map.svg", format='svg')
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
plt.matshow(matrix_to_plot.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
plt.colorbar()
plt.savefig("result_example_map.svg", format='svg')
plt.show()